
# Feature-family enrichment with decoupler

**The idea.** In transcriptomics you rarely interpret 20,000 genes one at a time; you
score *sets* of them — pathways, regulons, signatures — and interpret those. Cell
Painting has the same problem (2,776 features, most of them near-duplicates) and
`adata.var` already carries the structure needed to solve it the same way:

| gene-set analysis | Cell Painting analogue |
| --- | --- |
| gene | CellProfiler feature |
| gene set / pathway | feature **family** (`Texture`, `Granularity`, ...) |
| regulon / TF | **compartment** (`Nuclei`, `Cells`, `Cytoplasm`) or **channel** (`DNA`, `ER`, `AGP`, `Mito`) |
| pathway activity score | morphological **programme** score per well |

That turns "feature `Cyto_Texture_Contrast_Mito_5_02_256` went up" into "mitochondrial
texture increased in the cytoplasm", which is something you can reason about.

**Where the analogy breaks — read this before interpreting anything.** A curated gene set
encodes *biological* knowledge: these genes act together. A feature family encodes
*measurement* taxonomy: these numbers were computed by the same algorithm. A high
`Texture` score means "texture changed", not "a texture pathway was activated". The
scores are a legible summary, not a mechanistic claim.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False, figsize=(4, 4))

# Override with EU_OS_H5AD if your copy lives elsewhere.
H5AD = Path(os.environ.get("EU_OS_H5AD", "../data/eu_os_imtm_hepg2.h5ad"))
CLIP = 10.0  # see the heavy-tail section of notebook 01

import decoupler as dc

print("decoupler", dc.__version__ if hasattr(dc, "__version__") else "2.x")

In [ ]:

adata = ad.read_h5ad(H5AD)
adata.layers["robust_z"] = adata.X.copy()
adata.X = np.clip(adata.X, -CLIP, CLIP)  # see notebook 01, section 4
adata


## 2. Build the network from `var`

`decoupler` wants a long "network" frame: a `source` (the set) and a `target` (a member),
optionally weighted. Ours comes straight out of `adata.var` — no external database
involved, which is both the strength (nothing to download, nothing to map) and the
limitation (no biological curation) of this approach.

In [ ]:

from cellpainting_scverse import feature_sets, summarize

net = feature_sets(adata, keys=("compartment", "family", "channel"), min_size=5)
print(f"{len(net)} edges, {net.source.nunique()} sets")
print(summarize(net).to_string())

In [ ]:

# Equivalent by hand, if you would rather not depend on the helper:
rows = []
for key in ("compartment", "family", "channel"):
    values = adata.var[key].astype(object)
    for value, members in adata.var.groupby(values, observed=True).groups.items():
        if pd.isna(value):
            continue
        rows.append(pd.DataFrame({"source": f"{key}:{value}", "target": list(members)}))
net_manual = pd.concat(rows, ignore_index=True).assign(weight=1.0)
print(f"{len(net_manual)} edges, {net_manual.source.nunique()} sets")


### Audit the set sizes before you go further

Enrichment statistics are sensitive to set size, and these sets are grossly unbalanced.

In [ ]:

pruned = dc.pp.prune(features=adata.var_names.to_numpy(), net=net, tmin=5, verbose=True)
print(f"\nsets surviving tmin=5: {pruned.source.nunique()} of {net.source.nunique()}")
print("\ndropped:", sorted(set(net.source) - set(pruned.source)))


Nothing is dropped here, because `feature_sets(min_size=5)` already removed the only set
that was too small: `family:ObjectSkeleton`, which has exactly one feature. That is why the
helper reports 14 sets and the manual construction 15. Keeping `dc.pp.prune` in the flow is
still worth it — it is the step that guarantees the network and the matrix agree, and it
will drop sets if you subset features later.

The remaining imbalance is severe: `compartment:*` sets hold ~900 features each,
`family:Texture` 1,872, and `family:Neighbors` only 12. Two consequences:

- **Large sets are statistically easy and biologically vague.** A significant
  `family:Texture` score is nearly guaranteed and says little.
- **The informative sets are the specific ones.** Cross the keys to get them.

> **Suggestion.** Use crossed sets (`Nuclei|Texture|DNA`) as your primary analysis and the
> marginal sets only as a coarse summary. `feature_sets(..., combine=True)` adds them.

In [ ]:

net_x = feature_sets(adata, keys=("compartment", "family", "channel"), combine=True, min_size=10)
crossed = net_x[net_x.source.str.startswith("combined:")]
print(f"{crossed.source.nunique()} crossed sets with >=10 features")
print(summarize(crossed).head(12).to_string())


## 3. Choosing a method

`decoupler` ships a dozen estimators, and their assumptions differ in ways that matter
here. Our matrix is **signed, continuous, dense, roughly symmetric, and already
z-scored against controls** — which rules several of them out.

| method | what it assumes | verdict for Cell Painting |
| --- | --- | --- |
| `zscore` | values are comparable across features; mean of a set is meaningful | **best fit** — the matrix is literally already z-scores |
| `ulm` / `mlm` | linear model of the profile on set membership | **good** — sign-aware, handles weights |
| `waggr` | weighted aggregation | fine, needs weights you don't have |
| `gsea` | a *ranking* per observation | works, sign-aware, but slow at 10k observations |
| `ora` | a *selected* set (hypergeometric on a shortlist) | needs an arbitrary threshold, discards sign and magnitude |
| `aucell` | high values = "on"; built for sparse counts | **poor fit** — ranks only, and negatives are meaningful here |
| `gsva` | expression follows a kernel-estimable distribution | **poor fit** — designed for expression, heavy machinery for no gain |
| `viper` | signed regulons with confidence weights | no analogue for our unweighted sets |

We use `zscore` as the primary estimator and `ulm` as a cross-check.

> **Note on the null.** All of these assume set members are (roughly) independent.
> Texture features across scales and angles are anything but — correlations above 0.9 are
> routine. That inflates significance, so the p-values are optimistic. Section 5 builds
> an empirical null instead of trusting them.

## 4. Score the programmes

In [ ]:

dc.mt.zscore(adata, pruned, tmin=5, verbose=True)
print("\nnew obsm keys:", [k for k in adata.obsm if "zscore" in k])

scores = dc.pp.get_obsm(adata, "score_zscore")
print(scores)

In [ ]:

# ulm as a cross-check: do the two estimators agree on the ordering?
dc.mt.ulm(adata, pruned, tmin=5, verbose=False)


def as_frame(adata, key):
    obsm = dc.pp.get_obsm(adata, key)
    return pd.DataFrame(obsm.X, index=adata.obs_names, columns=obsm.var_names)


z = as_frame(adata, "score_zscore")
u = as_frame(adata, "score_ulm")
agree = pd.Series({c: np.corrcoef(z[c], u[c])[0, 1] for c in z.columns})
print("per-programme correlation between zscore and ulm:")
print(agree.round(3).sort_values().to_string())


Every correlation comes out at 1.000, which is not a bug and is worth understanding.
With **unweighted** sets and no covariates, fitting a univariate linear model of the
profile on a binary membership vector (`ulm`) is a monotone reparameterisation of taking
the mean of the set members (`zscore`) — the two differ by a scale factor, so they rank
observations identically.

The practical lesson: **`ulm` buys you nothing over `zscore` until your sets carry
weights.** If you later derive weights (say, each feature's loading on a positive-control
axis, as suggested in section 8), the two will diverge and `ulm` becomes the better choice.


## 5. Interpretation: do the positive controls behave?

The 20 annotated tubulin binders are our ground truth. Tubulin disruption arrests cells
in mitosis and reorganises the cytoskeleton, so it should be visible as a coherent shift
in nuclear and cytoskeletal programmes — not as a uniform increase everywhere.

In [ ]:

z["_group"] = np.select(
    [adata.obs.pert_type.eq("negcon").to_numpy(), adata.obs.tubulin_binder.to_numpy()],
    ["DMSO", "tubulin"], default="other",
)
summary = z.groupby("_group").mean().T
summary["tubulin_minus_dmso"] = summary["tubulin"] - summary["DMSO"]
print(summary.reindex(summary.tubulin_minus_dmso.abs().sort_values(ascending=False).index).round(2).to_string())


Expect `channel:DNA` down by roughly 4 z-units and `family:Intensity` down by ~3, with
`channel:AGP`, `family:RadialDistribution` and `family:AreaShape` up by 1.5-2.5.

That reads as: DNA-channel intensity drops while cells enlarge and signal redistributes
towards the periphery — consistent with mitotic arrest and cytoskeletal collapse. Note
this is a *coherent, signed, interpretable* pattern, which is the whole point of scoring
families rather than 2,776 individual features.

> **Careful.** DMSO scores are not exactly zero even though the data were normalized
> against DMSO, because the sets mix features whose residual offsets do not cancel. Always
> compare a group to DMSO, never to zero.

In [ ]:

dc.pl.barplot(summary.T, name="tubulin", top=14, vertical=True)


## 6. An empirical null: shuffle the network

Because features within a family are strongly correlated, the analytic p-values in
`obsm["padj_zscore"]` are too small. The cheap fix is to rebuild the sets at random,
preserving their sizes, and see how large a "tubulin vs DMSO" difference arises by chance.

In [ ]:

rng_shifts = []
for seed in range(20):
    shuffled = dc.pp.shuffle_net(pruned, target=True, seed=seed)
    tmp = adata.copy()
    dc.mt.zscore(tmp, shuffled, tmin=5, verbose=False)
    s = as_frame(tmp, "score_zscore")
    delta = s[adata.obs.tubulin_binder.to_numpy()].mean() - s[adata.obs.pert_type.eq("negcon").to_numpy()].mean()
    rng_shifts.append(delta)

null = pd.concat(rng_shifts, axis=1)
observed = summary["tubulin_minus_dmso"]
comparison = pd.DataFrame({
    "observed": observed,
    "null_mean": null.mean(axis=1),
    "null_sd": null.std(axis=1),
})
comparison["z_vs_null"] = (comparison.observed - comparison.null_mean) / comparison.null_sd.replace(0, np.nan)
print(comparison.round(2).to_string())


> **TODO.** 20 shuffles is enough to see the shape of the null but not to get a stable
> p-value; raise it if you want to report one. If a programme's `z_vs_null` is small, its
> apparent shift is a size artifact rather than biology.


## 7. Compound-level programmes

Scores are per well, but the unit of biological interest is the compound. Aggregate the
4 replicates — with the **median**, not the mean, since single bad wells are common.

In [ ]:

z_only = z.drop(columns="_group")
z_only["EOS"] = adata.obs.EOS.astype(str).to_numpy()
by_cmp = z_only.groupby("EOS").median()

meta = adata.obs.assign(EOS=lambda f: f.EOS.astype(str)).drop_duplicates("EOS").set_index("EOS")
by_cmp = by_cmp.join(meta[["compound", "tubulin_binder", "approved_drug", "pert_type"]])
print(f"{by_cmp.shape[0]} compounds x {z_only.shape[1] - 1} programmes")

prog = "channel:DNA"  # TODO: pick the programme you care about
print(f"\nmost extreme compounds on {prog}:")
print(by_cmp.reindex(by_cmp[prog].abs().sort_values(ascending=False).index)
      [[prog, "compound", "tubulin_binder"]].head(12).round(2).to_string())


Inspect that list sceptically. The most extreme compounds on `channel:DNA` typically come
back as things like palmidrol, sotalol and milrinone — pharmacologically inert in a
hepatocyte line at 10 uM, and none of them tubulin binders. A programme score of |z| ~ 14
for an inert compound is a red flag, not a discovery: it is what a plate or position
artifact looks like after being averaged over four replicates that all sit at the *same*
well coordinate.

> **Always cross-check extremes against reproducibility.** A compound is interesting when
> it moves a programme *and* its replicates agree beyond what position alone explains —
> which is precisely what notebook 03 builds. Rank by programme score alone and you will
> mostly rediscover the plate layout.


## 8. What does *not* translate — and what to do instead

### Pathway databases are unusable as-is

`decoupler`'s built-in resources (`dc.op.collectri`, `dc.op.progeny`, `dc.op.hallmark`)
map **gene symbols** to sets. Our `var_names` are CellProfiler measurements. There is no
mapping, and none can be invented — a texture feature is not a gene. Do not try.

### But there *is* a gene axis in this dataset — on the other axis

`adata.obs.target_genes` holds each compound's annotated protein targets as a
semicolon-separated list of gene symbols, covering ~95% of the library. The analysis that
uses it is the **transpose** of the usual one: instead of asking "which feature sets moved
in this well", ask "do compounds sharing a target gene share a morphology?"

In [ ]:

# Sketch: compound-level target sets, scored over morphological programmes.
targets = (
    adata.obs.drop_duplicates("EOS")
    .assign(EOS=lambda d: d.EOS.astype(str))
    .set_index("EOS")["target_genes"].astype(str)
)
target_net = (
    targets[targets.str.len() > 0]
    .str.split(";").explode().rename("source").reset_index()
    .rename(columns={"EOS": "target"})[["source", "target"]]
    .assign(weight=1.0)
)
sizes = target_net.groupby("source").size().sort_values(ascending=False)
print(f"{target_net.source.nunique()} target genes; {(sizes >= 5).sum()} with >=5 compounds")
print(sizes.head(10).to_string())


> **Filter these annotations before using them.** The largest sets are not mechanisms.
> `CYP3A4` (~380 compounds), `CYP1A2` and `ALDH1A1` are drug-metabolising enzymes that
> nearly every small molecule touches; `LMNA` (~380) and `MAPT` (~330) reflect broad
> binding-panel screens rather than primary pharmacology. Sets like these will look
> "enriched" for trivial reasons because they are close to a random sample of the library.
>
> Keep sets in a middle size band — roughly 5 to 50 compounds — and prefer genes that are a
> compound's *annotated primary target*. `TUBB`/`TUBA4A` are the sanity check: they should
> score highly, since the positive controls are in them.

To run it, transpose the compound-level programme matrix so that **compounds are the
variables** and target genes are the sets, then apply the same estimator. Concretely:
build an AnnData whose `obs` is programmes (or PCs) and whose `var` is compounds, and pass
`target_net`. Genes whose compounds share a morphology are candidate morphology-linked
targets.

> **TODO.** Restrict to target genes with >= 5 annotated compounds, and use
> `pt.tl.Enrichment` (notebook 03) or `dc.mt.ulm` on the transposed matrix.

### Other things worth knowing

- **No directional weights.** Curated regulons carry +1/-1 to say whether a target goes up
  or down. Feature families have no intrinsic direction, so `viper` and weighted `waggr`
  lose their advantage. You *could* derive weights empirically — e.g. each feature's
  loading on a positive-control axis — which effectively turns a family into a signature.
- **Consider real signatures instead.** The most valuable extension is to stop using
  measurement taxonomy and build *phenotypic* signatures: take the features that
  consistently move under a known MOA (tubulin binders here) and treat that as a set.
  Scoring the library against it is guilt-by-association MOA prediction, and it is a much
  closer analogue of a curated gene set than `family:Texture` will ever be.
- **Link to transcriptomics.** Many EU-OS bioactives are also in LINCS/L1000. Joining on
  `inchikey` gives a genuine gene axis for the same compounds, and then the entire
  pathway-database machinery applies — to the expression side, with morphology as the
  independent readout.


## Key takeaways

1. **`var` annotation is what makes this possible.** `compartment` x `family` x `channel`
   turns 2,776 opaque features into ~14 legible programmes with no external database.
2. **Use `zscore` or `ulm`.** The matrix is already z-scored against controls, which is
   exactly what these assume. `aucell` and `gsva` assume expression-like data and do not fit.
3. **Prefer crossed sets.** Marginal sets are huge and vague; `Nuclei|Texture|DNA` is
   specific enough to interpret.
4. **Distrust the analytic p-values.** Features within a family are highly correlated, so
   significance is inflated. Shuffle the network for an empirical null.
5. **A family score is descriptive, not mechanistic.** "Mitochondrial texture changed" is
   a legible summary; it is not a pathway activation claim.
6. **The pathway databases do not apply to `var`** — but `obs.target_genes` gives a real
   gene axis on the compound side, and that is where curated biology re-enters.

## References

- [decoupler documentation](https://decoupler.readthedocs.io/) — Badia-i-Mompel et al. 2022, https://doi.org/10.1093/bioadv/vbac016
- [Pathway analysis chapter, single-cell best practices](https://www.sc-best-practices.org/conditions/gsea_pathway.html)
- Bray et al. 2016, *Cell Painting assay protocol* — https://doi.org/10.1038/nprot.2016.105